# AMCNA Phase 3: Benchmark en Kaggle (GPU T4)
**Instrucción importante:** En la barra lateral derecha de Kaggle, asegúrate de cambiar **Accelerator** a **GPU T4 x2** antes de ejecutar.

In [ ]:
!pip install ncps scikit-learn scipy matplotlib numpy

import os
import sys

# Crear todas las carpetas necesarias
dirs = ["utils", "datasets", "models", "trainers", "experiments", "outputs", "figures", "data"]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("✅ Carpetas creadas correctamente.")



In [ ]:
# Escribiendo utils/seed.py
with open("utils/seed.py", "w", encoding="utf-8") as f:
    f.write("import os\nimport random\nimport numpy as np\nimport torch\n\ndef set_seed(seed: int = 42) -> None:\n    \"\"\"Fija la semilla en todos los m\u00f3dulos para reproducibilidad absoluta.\"\"\"\n    random.seed(seed)\n    os.environ['PYTHONHASHSEED'] = str(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    \n    if torch.cuda.is_available():\n        torch.cuda.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n    \n    # Configuraciones deterministas\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n    \n    # Utilizar algoritmos deterministas en PyTorch\n    torch.use_deterministic_algorithms(True, warn_only=True)\n")
print("Escrito: utils/seed.py")


In [ ]:
# Escribiendo utils/metrics.py
with open("utils/metrics.py", "w", encoding="utf-8") as f:
    f.write("import torch\nimport torch.nn as nn\nimport time\n\ndef count_parameters(model: nn.Module) -> int:\n    \"\"\"Devuelve el n\u00famero total de par\u00e1metros entrenables.\"\"\"\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\ndef match_parameters_by_hidden_dim(\n    create_model_fn, \n    target_params: int, \n    dim_range: range = range(2, 256), \n    tolerance: float = 0.05\n):\n    \"\"\"\n    Busca heur\u00edsticamente la dimensi\u00f3n oculta (d) que hace que el modelo creado\n    por `create_model_fn(d)` tenga un n\u00famero de par\u00e1metros dentro de un +-5% \n    respecto a `target_params`.\n    \"\"\"\n    best_d = None\n    best_diff = float('inf')\n    best_model = None\n    \n    for d in dim_range:\n        try:\n            model = create_model_fn(d)\n        except Exception:\n            continue\n            \n        params = count_parameters(model)\n        diff = abs(params - target_params)\n        \n        if diff < best_diff:\n            best_diff = diff\n            best_d = d\n            best_model = model\n            \n        # Si la diferencia es de crecimiento mon\u00f3tono y ya pasamos el target\n        if params > target_params * 1.5:\n            break\n            \n    if best_diff / target_params > tolerance:\n        print(f\"[WARNING] No se pudo emparejar par\u00e1metros en < {tolerance*100}%. \"\n              f\"Target: {target_params}, Best: {count_parameters(best_model)} (d={best_d})\")\n    \n    return best_model, best_d\n\nclass ComputationalCostTracker:\n    def __init__(self):\n        self.start_time = None\n        self.end_time = None\n        \n    def start(self):\n        self.start_time = time.time()\n        \n    def stop(self):\n        self.end_time = time.time()\n        return self.end_time - self.start_time\n        \n    def estimate_macs_rnn(self, model: nn.Module, seq_len: int, batch_size: int, input_dim: int, hidden_dim: int):\n        \"\"\"\n        Estimaci\u00f3n simple de MACs (Multiply-Accumulates) para la estructura de la Fase 1.\n        Dado que PtFlops u otras librer\u00edas pueden fallar en bucles for internos de RNNs custom,\n        hacemos un recuento basal. A mejorar si se requiere precisi\u00f3n exacta de hardware.\n        \"\"\"\n        # Para redes donde matrix mul domina, MACs ~ params * seq_len * batch_size.\n        # Es una heur\u00edstica de primer orden \u00fatil para Sanity Checks CPU.\n        params = count_parameters(model)\n        return params * seq_len * batch_size\n")
print("Escrito: utils/metrics.py")


In [ ]:
# Escribiendo datasets/smnist.py
with open("datasets/smnist.py", "w", encoding="utf-8") as f:
    f.write("import torch\nfrom torch.utils.data import DataLoader\nfrom torchvision import datasets, transforms\nimport numpy as np\n\nclass SequentialMNIST(torch.utils.data.Dataset):\n    \"\"\"\n    Sequential MNIST: Each 28x28 image is flattened to a sequence of 784 timesteps,\n    each with 1 feature (pixel intensity normalized to [0,1]).\n    This is the standard benchmark for evaluating RNNs on long sequences.\n    \"\"\"\n    def __init__(self, train=True, permutation=None, data_dir='./data'):\n        self.mnist = datasets.MNIST(data_dir, train=train, download=True, \n                                     transform=transforms.ToTensor())\n        self.permutation = permutation  # Fixed permutation for psMNIST\n        \n    def __len__(self):\n        return len(self.mnist)\n    \n    def __getitem__(self, idx):\n        img, label = self.mnist[idx]\n        # img: (1, 28, 28) -> flatten to (784,) -> reshape to (784, 1)\n        seq = img.view(-1)  # (784,)\n        if self.permutation is not None:\n            seq = seq[self.permutation]\n        seq = seq.unsqueeze(-1)  # (784, 1)\n        return seq, label  # label is an integer 0-9\n\n\ndef get_smnist_dataloaders(batch_size=128, permuted=False, data_dir='./data', num_workers=0):\n    \"\"\"\n    Returns (train_loader, val_loader, test_loader) for sMNIST or psMNIST.\n    Splits the 60k training set into 50k train + 10k validation.\n    Test set is the standard 10k.\n    \"\"\"\n    perm = None\n    if permuted:\n        # Fixed permutation, deterministic for reproducibility\n        rng = np.random.RandomState(1999)\n        perm = torch.from_numpy(rng.permutation(784)).long()\n    \n    train_full = SequentialMNIST(train=True, permutation=perm, data_dir=data_dir)\n    test_ds = SequentialMNIST(train=False, permutation=perm, data_dir=data_dir)\n    \n    # Split train into train (50k) + val (10k)\n    train_ds, val_ds = torch.utils.data.random_split(\n        train_full, [50000, 10000],\n        generator=torch.Generator().manual_seed(42)\n    )\n    \n    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)\n    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)\n    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)\n    \n    return train_loader, val_loader, test_loader\n")
print("Escrito: datasets/smnist.py")


In [ ]:
# Escribiendo models/amcna.py
with open("models/amcna.py", "w", encoding="utf-8") as f:
    f.write("import torch\nimport torch.nn as nn\nfrom ncps.torch import CfCCell\n\nclass AMCNA(nn.Module):\n    \"\"\"\n    Adaptive Multi-Channel Neural Architecture (AMCNA).\n    Implementa 5 canales de comunicaci\u00f3n recurrente ortogonal y un bucle de deliberaci\u00f3n.\n    \"\"\"\n    def __init__(self, input_size: int, hidden_size: int, output_size: int = 1,\n                 delay_steps: int = 5, deliberation_steps: int = 1, mod_dim: int = 4):\n        super().__init__()\n        self.hidden_size = hidden_size\n        self.delay_steps = delay_steps\n        self.L = deliberation_steps\n        \n        # Proyecci\u00f3n de entrada del entorno\n        self.env_proj = nn.Linear(input_size, hidden_size)\n        \n        # Canales (K=5)\n        self.w1 = nn.Linear(hidden_size, hidden_size, bias=False) # C1: Identity\n        self.w2 = nn.Linear(hidden_size, hidden_size, bias=False) # C2: Integral\n        self.w3 = nn.Linear(hidden_size, hidden_size, bias=False) # C3: Delay\n        self.w4 = nn.Linear(hidden_size, hidden_size, bias=False) # C4: Derivative\n        \n        self.w5 = nn.Linear(hidden_size, hidden_size, bias=False) # C5: Gating linear part\n        self.v5 = nn.Linear(hidden_size, hidden_size, bias=True)  # C5: Gating non-linear part\n        \n        # Estado Modulador Global\n        self.mod_dim = mod_dim\n        self.w_s = nn.Linear(mod_dim, mod_dim, bias=False)\n        self.v_s = nn.Linear(hidden_size, mod_dim, bias=True)\n        \n        # Ganancias por canal a partir del modulador\n        self.u_k = nn.Linear(mod_dim, 5, bias=False)\n        \n        # Din\u00e1mica de nodo base (CfC)\n        # Recibe la suma de los canales como \"input\"\n        self.cfc = CfCCell(hidden_size, hidden_size)\n        \n        self.out_proj = nn.Linear(hidden_size, output_size)\n        \n        # Constantes de filtrado\n        self.alpha = 0.1  # Decay para el integrador (C2)\n        self.beta = 0.5   # Decay para el derivativo (C4)\n        \n    def forward(self, x, return_gains: bool = False):\n        # x: (batch, seq_len, input_size)\n        batch_size, seq_len, _ = x.shape\n        device = x.device\n        \n        # Estados internos\n        h = torch.zeros(batch_size, self.hidden_size, device=device)\n        h_prev = torch.zeros_like(h)\n        z = torch.zeros_like(h)\n        d = torch.zeros_like(h)\n        s = torch.zeros(batch_size, self.mod_dim, device=device)\n        \n        # Buffer de estados para el delay (C3) implementado como lista (evita operaciones inplace)\n        delay_buffer = [torch.zeros(batch_size, self.hidden_size, device=device) for _ in range(self.delay_steps)]\n        \n        outputs = []\n        gains_history = []\n        ts = torch.ones(batch_size, 1, device=device)\n        \n        for t in range(seq_len):\n            env_in = self.env_proj(x[:, t, :])\n            \n            # Bucle deliberativo interno\n            for l in range(self.L):\n                # 1. Actualizaci\u00f3n de estados secundarios funcionales\n                z = (1 - self.alpha) * z + self.alpha * h\n                \n                # Derivada filtrada (h - h_prev)\n                delta_h = h - h_prev\n                d = self.beta * d + (1 - self.beta) * delta_h\n                \n                # Retardo puro\n                h_delay = delay_buffer[0]\n                \n                # 2. Modulador global\n                s = torch.tanh(self.w_s(s) + self.v_s(h))\n                # Ganancias (batch_size, 5)\n                gains = torch.sigmoid(self.u_k(s)) \n                if return_gains and l == self.L - 1:\n                    gains_history.append(gains.detach())\n                \n                # 3. Procesamiento de Canales\n                c1 = self.w1(h)\n                c2 = self.w2(z)\n                c3 = self.w3(h_delay)\n                c4 = self.w4(d)\n                c5 = torch.sigmoid(self.v5(h)) * self.w5(h)\n                \n                # Mezcla modulada\n                c_total = (gains[:, 0:1] * c1 + \n                           gains[:, 1:2] * c2 + \n                           gains[:, 2:3] * c3 + \n                           gains[:, 3:4] * c4 + \n                           gains[:, 4:5] * c5)\n                \n                # 4. Inyecci\u00f3n del entorno (solo en el primer paso deliberativo o siempre? \n                # Generalmente se mantiene constante durante la deliberaci\u00f3n)\n                node_input = c_total + env_in\n                \n                h_prev = h.clone()\n                # 5. Ecuaci\u00f3n din\u00e1mica de nodo (CfC)\n                h, _ = self.cfc(node_input, h, ts)\n            \n            # Actualizar buffer de delay al final del paso del entorno\n            delay_buffer.pop(0)\n            delay_buffer.append(h)\n            \n            outputs.append(h.unsqueeze(1))\n            \n        outputs = torch.cat(outputs, dim=1)\n        if return_gains:\n            # gains_history: (batch_size, seq_len, 5)\n            return self.out_proj(outputs), torch.stack(gains_history, dim=1)\n        return self.out_proj(outputs)\n\ndef create_amcna(hidden_dim: int, input_size: int = 1, output_size: int = 1, deliberation_steps: int = 1):\n    return AMCNA(input_size, hidden_dim, output_size, deliberation_steps=deliberation_steps)\n")
print("Escrito: models/amcna.py")


In [ ]:
# Escribiendo models/amcna_v2.py
with open("models/amcna_v2.py", "w", encoding="utf-8") as f:
    f.write("import torch\nimport torch.nn as nn\nfrom ncps.torch import CfCCell\n\nclass AMCNA_v2(nn.Module):\n    def __init__(self, input_size: int, hidden_size: int, output_size: int = 1,\n                 delay_steps: int = 5, deliberation_steps: int = 1):\n        super().__init__()\n        self.hidden_size = hidden_size\n        self.delay_steps = delay_steps\n        self.L = deliberation_steps\n        self.env_proj = nn.Linear(input_size, hidden_size)\n        self.w1 = nn.Linear(hidden_size, hidden_size, bias=False) # C1: Identity\n        self.w2 = nn.Linear(hidden_size, hidden_size, bias=False) # C2: Integral\n        self.w3 = nn.Linear(hidden_size, hidden_size, bias=False) # C3: Delay\n        self.w4 = nn.Linear(hidden_size, hidden_size, bias=False) # C4: Derivative\n        self.w5 = nn.Linear(hidden_size, hidden_size, bias=False) # C5: Gating linear\n        self.v5 = nn.Linear(hidden_size, hidden_size, bias=True)  # C5: Gating nonlinear\n\n        # Change A: Independent gain heads\n        self.gain_heads = nn.ModuleList([\n            nn.Sequential(nn.Linear(hidden_size, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())\n            for _ in range(5)\n        ])\n\n        self.cfc = CfCCell(hidden_size, hidden_size)\n        self.out_proj = nn.Linear(hidden_size, output_size)\n        self.alpha = 0.1\n        self.beta = 0.5\n        \n    def forward(self, x):\n        batch_size, seq_len, _ = x.shape\n        device = x.device\n        h = torch.zeros(batch_size, self.hidden_size, device=device)\n        h_prev = torch.zeros_like(h)\n        z = torch.zeros_like(h)\n        d = torch.zeros_like(h)\n        delay_buffer = [torch.zeros(batch_size, self.hidden_size, device=device) for _ in range(self.delay_steps)]\n        outputs = []\n        gains_history = []\n        ts = torch.ones(batch_size, 1, device=device)\n        for t in range(seq_len):\n            env_in = self.env_proj(x[:, t, :])\n            for l in range(self.L):\n                z = (1 - self.alpha) * z + self.alpha * h\n                delta_h = h - h_prev\n                d = self.beta * d + (1 - self.beta) * delta_h\n                h_delay = delay_buffer[0]\n                \n                # Compute independent gains\n                gain_list = [head(h) for head in self.gain_heads]\n                gains = torch.cat(gain_list, dim=1) # (batch_size, 5)\n\n                if l == self.L - 1:\n                    gains_history.append(gains)\n                    \n                c1 = self.w1(h)\n                c2 = self.w2(z)\n                c3 = self.w3(h_delay)\n                c4 = self.w4(d)\n                c5 = torch.sigmoid(self.v5(h)) * self.w5(h)\n                c_total = (gains[:, 0:1] * c1 + gains[:, 1:2] * c2 + gains[:, 2:3] * c3 + gains[:, 3:4] * c4 + gains[:, 4:5] * c5)\n                node_input = c_total + env_in\n                h_prev = h.clone()\n                h, _ = self.cfc(node_input, h, ts)\n            delay_buffer.pop(0)\n            delay_buffer.append(h)\n            outputs.append(h.unsqueeze(1))\n        outputs = torch.cat(outputs, dim=1)\n        gains_history_tensor = torch.stack(gains_history, dim=1) # (batch, seq, 5)\n        return self.out_proj(outputs), gains_history_tensor\n\n    def decorrelation_loss(self, gains):\n        # gains: (batch, seq, 5)\n        b, s, c = gains.shape\n        gains_flat = gains.view(b * s, c)\n        \n        # Center the gains\n        gains_centered = gains_flat - gains_flat.mean(dim=0, keepdim=True)\n        \n        # Compute covariance matrix\n        cov = torch.matmul(gains_centered.t(), gains_centered) / (b * s - 1)\n        \n        # Compute standard deviations\n        std = torch.sqrt(torch.diag(cov))\n        std_matrix = torch.outer(std, std)\n        std_matrix = torch.clamp(std_matrix, min=1e-8)\n        \n        # Correlation matrix\n        corr = cov / std_matrix\n        \n        # Identity matrix\n        I = torch.eye(c, device=gains.device)\n        \n        # Frobenius norm of (corr - I)\n        loss = torch.norm(corr - I, p='fro')**2\n        return loss\n\ndef create_amcna_v2(hidden_dim: int, input_size: int = 1, output_size: int = 1, deliberation_steps: int = 1):\n    return AMCNA_v2(input_size, hidden_dim, output_size, deliberation_steps=deliberation_steps)\n")
print("Escrito: models/amcna_v2.py")


In [ ]:
# Escribiendo models/baselines.py
with open("models/baselines.py", "w", encoding="utf-8") as f:
    f.write("import torch\nimport torch.nn as nn\nfrom ncps.torch import CfCCell\n\nclass CfCBaseline(nn.Module):\n    \"\"\"\n    Baseline puro usando la implementaci\u00f3n original de NCPs (CfCCell).\n    Procesa secuencias paso a paso de manera id\u00e9ntica a como lo har\u00eda AMCNA.\n    \"\"\"\n    def __init__(self, input_size: int, hidden_size: int, output_size: int = 1):\n        super().__init__()\n        self.hidden_size = hidden_size\n        self.cfc = CfCCell(input_size, hidden_size)\n        self.out_proj = nn.Linear(hidden_size, output_size)\n        \n    def forward(self, x):\n        batch_size, seq_len, _ = x.shape\n        hx = torch.zeros(batch_size, self.hidden_size, device=x.device)\n        \n        outputs = []\n        ts = torch.ones(batch_size, 1, device=x.device)\n        for t in range(seq_len):\n            hx, _ = self.cfc(x[:, t, :], hx, ts)\n            outputs.append(hx.unsqueeze(1))\n            \n        outputs = torch.cat(outputs, dim=1)\n        return self.out_proj(outputs)\n\n\nclass MLPBaseline(nn.Module):\n    \"\"\"\n    Baseline 'tonto' no-recurrente (MLP paso a paso / Time-Distributed).\n    Sirve para evaluar si la tarea requiere memoria temporal real.\n    \"\"\"\n    def __init__(self, input_size: int, hidden_size: int, output_size: int = 1):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Linear(input_size, hidden_size),\n            nn.ReLU(),\n            nn.Linear(hidden_size, hidden_size),\n            nn.ReLU(),\n            nn.Linear(hidden_size, output_size)\n        )\n        self.hidden_size = hidden_size\n        \n    def forward(self, x):\n        # x shape: (batch_size, seq_len, input_size)\n        return self.net(x)\n\n\n# Funciones helper para el emparejamiento de par\u00e1metros\ndef create_cfc_baseline(hidden_dim: int, input_size: int = 1, output_size: int = 1):\n    return CfCBaseline(input_size, hidden_dim, output_size)\n\ndef create_mlp_baseline(hidden_dim: int, input_size: int = 1, output_size: int = 1):\n    return MLPBaseline(input_size, hidden_dim, output_size)\n")
print("Escrito: models/baselines.py")


In [ ]:
# Escribiendo models/baselines_extended.py
with open("models/baselines_extended.py", "w", encoding="utf-8") as f:
    f.write("import torch\nimport torch.nn as nn\n\n\nclass LSTMBaseline(nn.Module):\n    \"\"\"LSTM baseline for sequence classification. Reads full sequence, predicts from last hidden state.\"\"\"\n    def __init__(self, input_size: int, hidden_size: int, output_size: int = 10, num_layers: int = 1):\n        super().__init__()\n        self.hidden_size = hidden_size\n        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)\n        self.out_proj = nn.Linear(hidden_size, output_size)\n        \n    def forward(self, x):\n        # x: (batch, seq_len, input_size)\n        output, (hn, cn) = self.lstm(x)\n        # Use last timestep output\n        return self.out_proj(output)  # (batch, seq_len, output_size) - trainer will slice last\n\n\nclass GRUBaseline(nn.Module):\n    \"\"\"GRU baseline for sequence classification.\"\"\"\n    def __init__(self, input_size: int, hidden_size: int, output_size: int = 10, num_layers: int = 1):\n        super().__init__()\n        self.hidden_size = hidden_size\n        self.gru = nn.GRU(input_size, hidden_size, num_layers=num_layers, batch_first=True)\n        self.out_proj = nn.Linear(hidden_size, output_size)\n        \n    def forward(self, x):\n        output, hn = self.gru(x)\n        return self.out_proj(output)\n\n\ndef create_lstm_baseline(hidden_dim: int, input_size: int = 1, output_size: int = 10):\n    return LSTMBaseline(input_size, hidden_dim, output_size)\n\ndef create_gru_baseline(hidden_dim: int, input_size: int = 1, output_size: int = 10):\n    return GRUBaseline(input_size, hidden_dim, output_size)\n")
print("Escrito: models/baselines_extended.py")


In [ ]:
# Escribiendo trainers/trainer.py
with open("trainers/trainer.py", "w", encoding="utf-8") as f:
    f.write("import torch\nimport torch.nn as nn\nimport time\nimport numpy as np\nfrom scipy import stats as scipy_stats\nfrom sklearn.metrics import auc\n\nclass ExperimentTrainer:\n    \"\"\"\n    Trainer con logging completo: convergencia, AUC, FLOPs, latencia,\n    memoria y estad\u00edsticos para comparaci\u00f3n cient\u00edfica.\n    \"\"\"\n    def __init__(self, model, train_loader, val_loader, test_loader, \n                 lr=3e-3, epochs=500, patience=50, device=\"cpu\"):\n        self.model = model.to(device)\n        self.train_loader = train_loader\n        self.val_loader = val_loader\n        self.test_loader = test_loader\n        self.epochs = epochs\n        self.patience = patience\n        self.device = device\n        \n        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)\n        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(\n            self.optimizer, factor=0.5, patience=15, min_lr=1e-5)\n        self.criterion = nn.BCEWithLogitsLoss()\n        \n    def _align_shapes(self, out, y):\n        if y.dim() == 2:\n            y = y.unsqueeze(-1)\n        elif y.dim() == 1:\n            y = y.view(-1, 1, 1)\n        if out.shape != y.shape:\n            out = out[:, -y.shape[1]:, :]\n        return out, y\n        \n    def train(self, verbose_every: int = 10):\n        best_val_loss = float('inf')\n        patience_counter = 0\n        best_val_acc = 0.0\n        best_epoch = 0\n        \n        history = {\n            \"train_loss\": [], \"val_loss\": [],\n            \"val_acc\": [], \"epoch_times\": [],\n            \"gains_per_epoch\": []  # Para curvas del modulador\n        }\n        \n        for epoch in range(self.epochs):\n            self.model.train()\n            train_loss = 0.0\n            t0 = time.perf_counter()\n            \n            for x, y in self.train_loader:\n                x, y = x.to(self.device), y.to(self.device)\n                self.optimizer.zero_grad()\n                out = self.model(x)\n                out, y = self._align_shapes(out, y)\n                loss = self.criterion(out, y)\n                loss.backward()\n                nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)\n                self.optimizer.step()\n                train_loss += loss.item()\n                \n            epoch_time = time.perf_counter() - t0\n            train_loss /= len(self.train_loader)\n            val_loss, val_acc = self.evaluate(self.val_loader)\n            self.scheduler.step(val_loss)\n            \n            history[\"train_loss\"].append(train_loss)\n            history[\"val_loss\"].append(val_loss)\n            history[\"val_acc\"].append(val_acc)\n            history[\"epoch_times\"].append(epoch_time)\n            \n            # Registrar ganancias de canal si el modelo las expone\n            if hasattr(self.model, 'forward') and 'return_gains' in self.model.forward.__code__.co_varnames:\n                self.model.eval()\n                with torch.no_grad():\n                    try:\n                        x_b, _ = next(iter(self.val_loader))\n                        x_b = x_b.to(self.device)\n                        _, g = self.model(x_b, return_gains=True)\n                        avg_g = g.mean(dim=(0, 1)).cpu().numpy()\n                        history[\"gains_per_epoch\"].append(avg_g.tolist())\n                    except Exception:\n                        pass\n                self.model.train()\n            \n            if val_acc > best_val_acc:\n                best_val_acc = val_acc\n                best_epoch = epoch\n                \n            if val_loss < best_val_loss:\n                best_val_loss = val_loss\n                patience_counter = 0\n            else:\n                patience_counter += 1\n                \n            if patience_counter >= self.patience:\n                if verbose_every > 0:\n                    print(f\"   [Early stop en \u00e9poca {epoch+1}]\", flush=True)\n                break\n                \n            if verbose_every > 0 and (epoch + 1) % verbose_every == 0:\n                print(f\"   Epoch {epoch+1}/{self.epochs} | \"\n                      f\"TL:{train_loss:.4f} VL:{val_loss:.4f} VA:{val_acc:.4f}\", flush=True)\n                \n        # === M\u00e9tricas finales ===\n        test_loss, test_acc = self.evaluate(self.test_loader)\n        \n        # AUC de la curva de accuracy (eficiencia de aprendizaje)\n        epochs_x = list(range(len(history[\"val_acc\"])))\n        if len(epochs_x) > 1:\n            val_acc_list = history[\"val_acc\"]\n            acc_auc = float(sum((val_acc_list[i] + val_acc_list[i+1]) / 2.0 for i in range(len(val_acc_list) - 1))) / (len(epochs_x) - 1)\n        else:\n            acc_auc = 0.0\n        \n        # Latencia de inferencia\n        lat_ms = self._measure_inference_latency()\n        \n        return {\n            \"history\": history,\n            \"test_loss\": test_loss,\n            \"test_acc\": test_acc,\n            \"best_val_acc\": best_val_acc,\n            \"best_epoch\": best_epoch,\n            \"auc_acc\": acc_auc,\n            \"inference_latency_ms\": lat_ms,\n            \"mean_epoch_time\": float(np.mean(history[\"epoch_times\"])),\n            \"epochs_run\": len(history[\"train_loss\"])\n        }\n        \n    def evaluate(self, loader):\n        self.model.eval()\n        total_loss = 0.0\n        correct = 0\n        total = 0\n        \n        with torch.no_grad():\n            for x, y in loader:\n                x, y = x.to(self.device), y.to(self.device)\n                out = self.model(x)\n                out, y = self._align_shapes(out, y)\n                loss = self.criterion(out, y)\n                total_loss += loss.item()\n                preds = (torch.sigmoid(out) > 0.5).float()\n                correct += (preds == y).sum().item()\n                total += y.numel()\n                \n        return total_loss / len(loader), correct / total if total > 0 else 0.0\n    \n    def _measure_inference_latency(self, n_runs: int = 20) -> float:\n        \"\"\"Mide latencia media de inferencia en ms.\"\"\"\n        self.model.eval()\n        x_b, _ = next(iter(self.test_loader))\n        x_b = x_b.to(self.device)\n        times = []\n        with torch.no_grad():\n            for _ in range(n_runs):\n                t0 = time.perf_counter()\n                self.model(x_b)\n                times.append((time.perf_counter() - t0) * 1000)\n        return float(np.mean(times[5:]))  # Descartar warm-up\n\n\ndef compute_statistics(values: list, reference: list = None) -> dict:\n    \"\"\"\n    Calcula estad\u00edsticos de un conjunto de valores.\n    Si se provee 'reference', calcula Cohen's d y p-value vs referencia.\n    \"\"\"\n    arr = np.array(values)\n    n = len(arr)\n    mean = float(arr.mean())\n    std = float(arr.std(ddof=1)) if n > 1 else 0.0\n    ci_half = float(1.96 * std / np.sqrt(n)) if n > 1 else 0.0\n    \n    result = {\n        \"mean\": mean,\n        \"std\": std,\n        \"ci_95\": (mean - ci_half, mean + ci_half),\n        \"n\": n\n    }\n    \n    if reference is not None and len(reference) > 1:\n        ref = np.array(reference)\n        # Cohen's d\n        pooled_std = np.sqrt((arr.std(ddof=1)**2 + ref.std(ddof=1)**2) / 2)\n        cohens_d = (mean - ref.mean()) / pooled_std if pooled_std > 0 else 0.0\n        \n        # Shapiro-Wilk para normalidad (decide entre t-test o Wilcoxon)\n        _, p_sw_arr = scipy_stats.shapiro(arr) if n >= 3 else (None, 0.0)\n        _, p_sw_ref = scipy_stats.shapiro(ref) if len(ref) >= 3 else (None, 0.0)\n        \n        if p_sw_arr > 0.05 and p_sw_ref > 0.05:\n            _, p_val = scipy_stats.ttest_ind(arr, ref)\n            test_used = \"t-test\"\n        else:\n            _, p_val = scipy_stats.mannwhitneyu(arr, ref, alternative='two-sided')\n            test_used = \"Mann-Whitney\"\n            \n        result[\"cohens_d\"] = float(cohens_d)\n        result[\"p_value\"] = float(p_val)\n        result[\"test_used\"] = test_used\n        \n    return result\n")
print("Escrito: trainers/trainer.py")


In [ ]:
# Escribiendo experiments/run_phase3_hpo.py
with open("experiments/run_phase3_hpo.py", "w", encoding="utf-8") as f:
    f.write("import os\nimport sys\nimport json\nimport torch\nimport torch.nn as nn\nimport torch.optim as optim\nfrom torch.utils.data import DataLoader\nfrom torchvision import datasets, transforms\nimport matplotlib\nmatplotlib.use('Agg')\n\n# Insert project root into path\nPROJECT_ROOT = os.path.abspath(\".\")\nsys.path.insert(0, PROJECT_ROOT)\n\nfrom utils.seed import set_seed\nfrom models.amcna_v2 import AMCNA_v2\nfrom ncps.torch import CfCCell\n\nclass CfCClassifier(nn.Module):\n    def __init__(self, hidden_size, input_size=1, output_size=10):\n        super().__init__()\n        self.hidden_size = hidden_size\n        self.cfc = CfCCell(input_size, hidden_size)\n        self.out_proj = nn.Linear(hidden_size, output_size)\n        \n    def forward(self, x):\n        batch_size, seq_len, _ = x.shape\n        hx = torch.zeros(batch_size, self.hidden_size, device=x.device)\n        ts = torch.ones(batch_size, 1, device=x.device)\n        for t in range(seq_len):\n            hx, _ = self.cfc(x[:, t, :], hx, ts)\n        return self.out_proj(hx).unsqueeze(1)  # (batch, 1, 10)\n\nclass AMCNA_v2_Classifier(nn.Module):\n    def __init__(self, hidden_size, input_size=1, output_size=10, deliberation_steps=1):\n        super().__init__()\n        self.net = AMCNA_v2(input_size, hidden_size, output_size, deliberation_steps=deliberation_steps)\n        \n    def forward(self, x):\n        out, gains = self.net(x)\n        return out[:, -1:, :], gains  # last timestep only\n\nclass LSTMClassifier(nn.Module):\n    def __init__(self, hidden_size, input_size=1, output_size=10):\n        super().__init__()\n        self.hidden_size = hidden_size\n        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)\n        self.out_proj = nn.Linear(hidden_size, output_size)\n        \n    def forward(self, x):\n        output, _ = self.lstm(x)\n        return self.out_proj(output[:, -1:, :])  # (batch, 1, 10)\n\nclass GRUClassifier(nn.Module):\n    def __init__(self, hidden_size, input_size=1, output_size=10):\n        super().__init__()\n        self.hidden_size = hidden_size\n        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)\n        self.out_proj = nn.Linear(hidden_size, output_size)\n        \n    def forward(self, x):\n        output, _ = self.gru(x)\n        return self.out_proj(output[:, -1:, :])  # (batch, 1, 10)\n\n\ndef get_model(model_name, hidden_size):\n    if model_name == \"CfC\":\n        return CfCClassifier(hidden_size)\n    elif model_name == \"AMCNA_v2\":\n        return AMCNA_v2_Classifier(hidden_size)\n    elif model_name == \"LSTM\":\n        return LSTMClassifier(hidden_size)\n    elif model_name == \"GRU\":\n        return GRUClassifier(hidden_size)\n    else:\n        raise ValueError(f\"Unknown model {model_name}\")\n\ndef train_and_eval(model, train_loader, val_loader, lr, epochs=15, patience=8):\n    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n    model.to(device)\n    \n    optimizer = optim.Adam(model.parameters(), lr=lr)\n    criterion = nn.CrossEntropyLoss()\n    \n    best_val_acc = 0.0\n    patience_counter = 0\n    \n    for epoch in range(epochs):\n        model.train()\n        for x, y in train_loader:\n            x = x.view(-1, 784, 1).to(device)\n            y = y.to(device)\n            \n            optimizer.zero_grad()\n            out = model(x)\n            if isinstance(out, tuple):\n                out, gains = out\n                \n            logits = out[:, -1, :]\n            loss = criterion(logits, y)\n            loss.backward()\n            optimizer.step()\n            \n        model.eval()\n        correct = 0\n        total = 0\n        with torch.no_grad():\n            for x, y in val_loader:\n                x = x.view(-1, 784, 1).to(device)\n                y = y.to(device)\n                \n                out = model(x)\n                if isinstance(out, tuple):\n                    out, gains = out\n                    \n                logits = out[:, -1, :]\n                preds = logits.argmax(dim=1)\n                correct += (preds == y).sum().item()\n                total += y.size(0)\n                \n        val_acc = correct / total\n        if val_acc > best_val_acc:\n            best_val_acc = val_acc\n            patience_counter = 0\n        else:\n            patience_counter += 1\n            \n        if patience_counter >= patience:\n            break\n            \n    return best_val_acc\n\ndef main():\n    data_dir = os.path.join(PROJECT_ROOT, \"data\")\n    os.makedirs(data_dir, exist_ok=True)\n    \n    transform = transforms.Compose([transforms.ToTensor()])\n    \n    full_train_dataset = datasets.MNIST(root=data_dir, train=True, download=True, transform=transform)\n    # Split into train and val (e.g., 50k train, 10k val) for HPO\n    train_size = 50000\n    val_size = len(full_train_dataset) - train_size\n    train_dataset, val_dataset = torch.utils.data.random_split(\n        full_train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)\n    )\n    \n    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)\n    val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)\n\n    models_to_tune = [\"CfC\", \"AMCNA_v2\", \"LSTM\", \"GRU\"]\n    learning_rates = [1e-2, 3e-3, 1e-3, 3e-4]\n    hidden_dims = [64, 96, 128]\n    n_seeds = 3\n    \n    results = {}\n    best_configs = {}\n    \n    outputs_dir = os.path.join(PROJECT_ROOT, \"outputs\")\n    os.makedirs(outputs_dir, exist_ok=True)\n    \n    for model_name in models_to_tune:\n        print(f\"--- Tuning {model_name} ---\")\n        results[model_name] = {}\n        best_mean_acc = 0.0\n        best_cfg = None\n        \n        for hd in hidden_dims:\n            for lr in learning_rates:\n                cfg_name = f\"hd_{hd}_lr_{lr}\"\n                print(f\"  Config: {cfg_name}\")\n                accs = []\n                for seed in range(n_seeds):\n                    set_seed(42 + seed)\n                    model = get_model(model_name, hd)\n                    val_acc = train_and_eval(model, train_loader, val_loader, lr)\n                    accs.append(val_acc)\n                \n                mean_acc = sum(accs) / len(accs)\n                results[model_name][cfg_name] = {\n                    \"hidden_dim\": hd,\n                    \"lr\": lr,\n                    \"accs\": accs,\n                    \"mean_acc\": mean_acc\n                }\n                print(f\"    Mean Val Acc: {mean_acc:.4f} {accs}\")\n                \n                if mean_acc > best_mean_acc:\n                    best_mean_acc = mean_acc\n                    best_cfg = results[model_name][cfg_name]\n                    \n        best_configs[model_name] = best_cfg\n        print(f\"  Best {model_name}: {best_cfg['hidden_dim']} hd, {best_cfg['lr']} lr (Acc: {best_mean_acc:.4f})\")\n        \n    print(\"\\n=== Best Configurations ===\")\n    for model_name, cfg in best_configs.items():\n        print(f\"{model_name:10s} | HD: {cfg['hidden_dim']:3d} | LR: {cfg['lr']:.4e} | Val Acc: {cfg['mean_acc']:.4f}\")\n        \n    with open(os.path.join(outputs_dir, \"hpo_results.json\"), \"w\") as f:\n        json.dump({\"all_results\": results, \"best_configs\": best_configs}, f, indent=4)\n        \nif __name__ == \"__main__\":\n    main()\n")
print("Escrito: experiments/run_phase3_hpo.py")


In [ ]:
# Escribiendo experiments/run_phase3_main.py
with open("experiments/run_phase3_main.py", "w", encoding="utf-8") as f:
    f.write("import os\nimport sys\nimport json\nimport time\nimport math\nimport torch\nimport torch.nn as nn\nimport torch.optim as optim\nimport torchvision\nfrom torchvision import transforms\nimport numpy as np\nimport scipy.stats as stats\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\n\nPROJECT_ROOT = os.path.abspath(\".\")\nsys.path.insert(0, PROJECT_ROOT)\n\nfrom utils.seed import set_seed\nfrom utils.metrics import count_parameters\n\n# =======================\n# Model Wrappers\n# =======================\nclass CfCClassifier(nn.Module):\n    def __init__(self, hidden_size, input_size=1, output_size=10):\n        super().__init__()\n        from ncps.torch import CfCCell\n        self.hidden_size = hidden_size\n        self.cfc = CfCCell(input_size, hidden_size)\n        self.out_proj = nn.Linear(hidden_size, output_size)\n    def forward(self, x):\n        batch_size, seq_len, _ = x.shape\n        hx = torch.zeros(batch_size, self.hidden_size, device=x.device)\n        ts = torch.ones(batch_size, 1, device=x.device)\n        for t in range(seq_len):\n            hx, _ = self.cfc(x[:, t, :], hx, ts)\n        return self.out_proj(hx).unsqueeze(1)\n\nclass AMCNA_v2_Classifier(nn.Module):\n    def __init__(self, hidden_size, input_size=1, output_size=10, deliberation_steps=1):\n        super().__init__()\n        from models.amcna_v2 import AMCNA_v2\n        self.net = AMCNA_v2(input_size, hidden_size, output_size, deliberation_steps=deliberation_steps)\n    def forward(self, x):\n        out, gains = self.net(x)\n        return out[:, -1:, :], gains\n\nclass LSTMClassifier(nn.Module):\n    def __init__(self, hidden_size, input_size=1, output_size=10):\n        super().__init__()\n        self.hidden_size = hidden_size\n        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)\n        self.out_proj = nn.Linear(hidden_size, output_size)\n    def forward(self, x):\n        output, _ = self.lstm(x)\n        return self.out_proj(output[:, -1:, :])\n\nclass GRUClassifier(nn.Module):\n    def __init__(self, hidden_size, input_size=1, output_size=10):\n        super().__init__()\n        self.hidden_size = hidden_size\n        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)\n        self.out_proj = nn.Linear(hidden_size, output_size)\n    def forward(self, x):\n        output, _ = self.gru(x)\n        return self.out_proj(output[:, -1:, :])\n\n# =======================\n# Data Loading\n# =======================\ndef get_dataloaders(batch_size=128):\n    data_dir = os.path.join(PROJECT_ROOT, \"data\")\n    os.makedirs(data_dir, exist_ok=True)\n    \n    transform = transforms.Compose([\n        transforms.ToTensor(),\n        transforms.Lambda(lambda x: x.view(-1, 1))\n    ])\n    \n    train_dataset = torchvision.datasets.MNIST(root=data_dir, train=True, download=True, transform=transform)\n    test_dataset = torchvision.datasets.MNIST(root=data_dir, train=False, download=True, transform=transform)\n    \n    train_size = 5000\n    val_size = 1000\n    test_size = 1000\n    \n    # Deterministic subset for faster training\n    generator = torch.Generator().manual_seed(42)\n    \n    # We only take a subset from the datasets to drastically reduce time\n    train_dataset, _ = torch.utils.data.random_split(train_dataset, [train_size, len(train_dataset) - train_size], generator=generator)\n    val_dataset, _ = torch.utils.data.random_split(test_dataset, [val_size, len(test_dataset) - val_size], generator=generator)\n    test_dataset, _ = torch.utils.data.random_split(test_dataset, [test_size, len(test_dataset) - test_size], generator=torch.Generator().manual_seed(43))\n    \n    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)\n    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)\n    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)\n    \n    # Fixed permutation for psMNIST\n    np.random.seed(42)\n    perm = torch.from_numpy(np.random.permutation(784)).long()\n    \n    return train_loader, val_loader, test_loader, perm\n\n# =======================\n# Decorrelation Loss\n# =======================\ndef compute_decorrelation_loss(gains):\n    if len(gains.shape) == 3:\n        b, t, m = gains.shape\n        gains = gains.view(-1, m)\n    elif len(gains.shape) > 3:\n        gains = gains.view(-1, gains.shape[-1])\n        \n    mean = gains.mean(dim=0, keepdim=True)\n    centered = gains - mean\n    cov = torch.matmul(centered.t(), centered) / (centered.size(0) - 1 + 1e-8)\n    \n    var = torch.diag(cov)\n    std = torch.sqrt(var + 1e-8)\n    corr = cov / (torch.outer(std, std) + 1e-8)\n    \n    eye = torch.eye(corr.size(0), device=corr.device)\n    loss = torch.norm(corr - eye, p='fro')\n    return loss\n\n# =======================\n# Training Loop\n# =======================\ndef train_model(model, train_loader, val_loader, test_loader, perm, task, model_name, lr, device, seed):\n    set_seed(seed)\n    \n    optimizer = optim.Adam(model.parameters(), lr=lr)\n    criterion = nn.CrossEntropyLoss()\n    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)\n    \n    best_val_acc = 0.0\n    best_model_state = None\n    patience_counter = 0\n    patience = 8\n    epochs = 25\n    \n    val_acc_curve = []\n    \n    for epoch in range(epochs):\n        model.train()\n        train_loss = 0.0\n        \n        for x, y in train_loader:\n            if task == 'psmnist':\n                x = x[:, perm, :]\n            \n            x, y = x.to(device), y.to(device)\n            optimizer.zero_grad()\n            \n            out = model(x)\n            if isinstance(out, tuple):\n                logits, gains = out\n                logits = logits.squeeze(1)\n                loss_ce = criterion(logits, y)\n                loss_dec = compute_decorrelation_loss(gains)\n                loss = loss_ce + 0.1 * loss_dec\n            else:\n                logits = out.squeeze(1)\n                loss = criterion(logits, y)\n                \n            loss.backward()\n            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)\n            optimizer.step()\n            train_loss += loss.item()\n            \n        model.eval()\n        correct = 0\n        total = 0\n        with torch.no_grad():\n            for x, y in val_loader:\n                if task == 'psmnist':\n                    x = x[:, perm, :]\n                x, y = x.to(device), y.to(device)\n                \n                out = model(x)\n                if isinstance(out, tuple):\n                    logits, _ = out\n                else:\n                    logits = out\n                    \n                logits = logits.squeeze(1)\n                preds = logits.argmax(dim=-1)\n                correct += (preds == y).sum().item()\n                total += y.size(0)\n                \n        val_acc = correct / total\n        val_acc_curve.append(val_acc)\n        scheduler.step(val_acc)\n        \n        if val_acc > best_val_acc:\n            best_val_acc = val_acc\n            best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}\n            patience_counter = 0\n        else:\n            patience_counter += 1\n            \n        print(f\"Task: {task} | Model: {model_name} | Seed: {seed} | Epoch {epoch+1}/{epochs} - Val Acc: {val_acc:.4f} - LR: {optimizer.param_groups[0]['lr']:.6f}\")\n        \n        if patience_counter >= patience:\n            print(\"Early stopping triggered.\")\n            break\n            \n    # Test phase\n    model.load_state_dict(best_model_state)\n    model.eval()\n    correct = 0\n    total = 0\n    with torch.no_grad():\n        for x, y in test_loader:\n            if task == 'psmnist':\n                x = x[:, perm, :]\n            x, y = x.to(device), y.to(device)\n            out = model(x)\n            if isinstance(out, tuple):\n                logits, _ = out\n            else:\n                logits = out\n            logits = logits.squeeze(1)\n            preds = logits.argmax(dim=-1)\n            correct += (preds == y).sum().item()\n            total += y.size(0)\n            \n    test_acc = correct / total\n    \n    # Latency measurement\n    latency_measurements = []\n    dummy_x = torch.randn(1, 784, 1, device=device)\n    with torch.no_grad():\n        # Warm up\n        for _ in range(5):\n            model(dummy_x)\n        # Measure\n        for _ in range(20):\n            start = time.perf_counter()\n            model(dummy_x)\n            end = time.perf_counter()\n            latency_measurements.append((end - start) * 1000) # ms\n            \n    avg_latency = np.mean(latency_measurements[-15:])\n    \n    return test_acc, best_val_acc, val_acc_curve, avg_latency\n\n# =======================\n# Stats and Formatting\n# =======================\ndef compute_cohens_d(x, y):\n    nx = len(x)\n    ny = len(y)\n    dof = nx + ny - 2\n    pool_var = ((nx-1)*np.var(x, ddof=1) + (ny-1)*np.var(y, ddof=1)) / dof\n    d = (np.mean(x) - np.mean(y)) / np.sqrt(pool_var)\n    return d\n\ndef compute_statistics(task_results, ref_model='CfC'):\n    stats_dict = {}\n    ref_accs = task_results[ref_model]['test_accs']\n    \n    for m, r in task_results.items():\n        accs = r['test_accs']\n        mean_acc = np.mean(accs)\n        std_acc = np.std(accs, ddof=1)\n        ci_95 = stats.t.interval(0.95, len(accs)-1, loc=mean_acc, scale=stats.sem(accs))\n        lat = np.mean(r['latencies'])\n        \n        # Compare with ref\n        if m == ref_model:\n            d = float('nan')\n            p = float('nan')\n        else:\n            d = compute_cohens_d(accs, ref_accs)\n            # Shapiro-Wilk test for normality\n            _, p_shapiro_m = stats.shapiro(accs)\n            _, p_shapiro_ref = stats.shapiro(ref_accs)\n            \n            if p_shapiro_m > 0.05 and p_shapiro_ref > 0.05:\n                _, p = stats.ttest_ind(accs, ref_accs, equal_var=False)\n            else:\n                _, p = stats.mannwhitneyu(accs, ref_accs)\n                \n        stats_dict[m] = {\n            'mean_acc': mean_acc,\n            'std_acc': std_acc,\n            'ci': ci_95,\n            'd': d,\n            'p': p,\n            'lat': lat,\n            'params': r['params']\n        }\n    return stats_dict\n\ndef print_table(task_name, stats_dict, ref_model='CfC'):\n    print(f\"\\n=== {task_name} Results (N=5) ===\")\n    print(f\"{'Model':<12} | {'Params':<8} | {'Test Acc (Mean \u00b1 Std)':<22} | {'95% CI':<17} | {'vs CfC (Cohen d)':<19} | {'p-value':<7} | {'Latency (ms)':<12}\")\n    print(\"-\" * 115)\n    \n    for m, s in stats_dict.items():\n        ci_str = f\"[{s['ci'][0]:.4f}, {s['ci'][1]:.4f}]\"\n        if m == ref_model:\n            d_str = \"\u2014\"\n            p_str = \"\u2014\"\n        else:\n            d_str = f\"{s['d']:.2f}\"\n            p_str = f\"{s['p']:.3f}\"\n            \n        print(f\"{m:<12} | {s['params']:<8} | {s['mean_acc']:.4f} \u00b1 {s['std_acc']:.4f} | {ci_str:<17} | {d_str:<19} | {p_str:<7} | {s['lat']:.2f}\")\n\n# =======================\n# Main\n# =======================\ndef main():\n    device = torch.device('cpu')\n    \n    hpo_file = os.path.join(PROJECT_ROOT, \"outputs\", \"hpo_results.json\")\n    hpo_results = {}\n    if os.path.exists(hpo_file):\n        with open(hpo_file, 'r') as f:\n            hpo_results = json.load(f)\n            \n    models_dict = {\n        'CfC': CfCClassifier,\n        'AMCNA_v2': AMCNA_v2_Classifier,\n        'LSTM': LSTMClassifier,\n        'GRU': GRUClassifier\n    }\n    \n    tasks = ['sMNIST', 'psMNIST']\n    n_seeds = 5\n    \n    train_loader, val_loader, test_loader, perm = get_dataloaders(batch_size=128)\n    \n    results = {t: {m: {'test_accs': [], 'val_acc_curves': [], 'latencies': [], 'params': 0} for m in models_dict} for t in tasks}\n    \n    os.makedirs(os.path.join(PROJECT_ROOT, \"outputs\"), exist_ok=True)\n    os.makedirs(os.path.join(PROJECT_ROOT, \"figures\"), exist_ok=True)\n    \n    for task in tasks:\n        task_lower = task.lower()\n        for model_name, model_cls in models_dict.items():\n            # Get hyperparams\n            if model_name in hpo_results:\n                lr = hpo_results[model_name].get('lr', 1e-3)\n                hidden_dim = hpo_results[model_name].get('hidden_dim', 96)\n            else:\n                lr = 1e-3\n                hidden_dim = 96\n                \n            for seed in range(n_seeds):\n                # Init model\n                model = model_cls(hidden_dim, input_size=1, output_size=10).to(device)\n                if seed == 0:\n                    results[task][model_name]['params'] = count_parameters(model)\n                    \n                test_acc, best_val, curve, lat = train_model(\n                    model, train_loader, val_loader, test_loader, perm, task_lower, model_name, lr, device, seed\n                )\n                \n                results[task][model_name]['test_accs'].append(test_acc)\n                results[task][model_name]['val_acc_curves'].append(curve)\n                results[task][model_name]['latencies'].append(lat)\n                \n        # Stats & Table\n        stats_dict = compute_statistics(results[task])\n        print_table(task, stats_dict)\n        \n    # Plotting\n    fig, axes = plt.subplots(1, 2, figsize=(12, 5))\n    for i, task in enumerate(tasks):\n        ax = axes[i]\n        for model_name in models_dict:\n            curves = results[task][model_name]['val_acc_curves']\n            max_len = max(len(c) for c in curves)\n            # Pad with last value\n            padded_curves = []\n            for c in curves:\n                padded = c + [c[-1]] * (max_len - len(c))\n                padded_curves.append(padded)\n            \n            curves_arr = np.array(padded_curves)\n            mean_curve = curves_arr.mean(axis=0)\n            std_curve = curves_arr.std(axis=0)\n            x_vals = np.arange(1, max_len + 1)\n            \n            ax.plot(x_vals, mean_curve, label=model_name)\n            ax.fill_between(x_vals, mean_curve - std_curve, mean_curve + std_curve, alpha=0.2)\n            \n        ax.set_title(f\"{task} Validation Accuracy\")\n        ax.set_xlabel(\"Epoch\")\n        ax.set_ylabel(\"Accuracy\")\n        ax.legend()\n        ax.grid(True)\n        \n    plt.tight_layout()\n    plt.savefig(os.path.join(PROJECT_ROOT, \"figures\", \"phase3_curves.png\"))\n    \n    # Save results\n    with open(os.path.join(PROJECT_ROOT, \"outputs\", \"phase3_results.json\"), 'w') as f:\n        json.dump(results, f, indent=4)\n\nif __name__ == \"__main__\":\n    print(\"Starting Phase 3 Experiment...\")\n    start_time = time.perf_counter()\n    main()\n    end_time = time.perf_counter()\n    total_minutes = (end_time - start_time) / 60\n    print(f\"\\n\u2705 All tests finished. Total execution time: {total_minutes:.2f} minutes.\")\n")
print("Escrito: experiments/run_phase3_main.py")


In [ ]:
import sys
import os
import torch

sys.path.insert(0, os.path.abspath('.'))

print("==========================================")
print("GPU Disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo detectado:", torch.cuda.get_device_name(0))
else:
    print("⚠️ ADVERTENCIA: No se detectó GPU. Activa GPU T4 en la barra derecha de Kaggle (Accelerator -> GPU T4).")
print("==========================================")

# Crear dummy HPO para evitar la búsqueda lenta de hiperparámetros
import json
dummy_hpo = {
    "CfC": {"hidden_dim": 96, "lr": 0.003},
    "AMCNA_v2": {"hidden_dim": 96, "lr": 0.003},
    "LSTM": {"hidden_dim": 96, "lr": 0.003},
    "GRU": {"hidden_dim": 96, "lr": 0.003}
}
os.makedirs("outputs", exist_ok=True)
with open("outputs/hpo_results.json", "w") as f:
    json.dump(dummy_hpo, f)

# Ejecutar el experimento principal
!python experiments/run_phase3_main.py

